# RichPrompt Stage 1 — Visual–Text Alignment

Notebook riêng cho experiment **RichCount-inspired full-image alignment for T2ICount**.

Workflow:

1. Mount Google Drive.
2. Checkout branch `rich-prompt-alignment`.
3. Cài dependencies.
4. Chuẩn bị asset local + offline CLIP processor config.
5. Chạy `--validate-only`.
6. Chạy smoke: 1 batch FFN + 1 batch adapter.
7. Đọc `alignment_summary.json` và kiểm tra normalized-distance behavior.
8. **Chỉ chạy full 10+10 epochs sau khi normalized smoke được kiểm tra.**

Notebook này **không train counting model**, không dùng DUMLO và không thay đổi RichPrompt Phase 2.


In [1]:
# 1. Runtime / GPU check
import os, sys, subprocess, json
from pathlib import Path

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# 3. Configuration
from pathlib import Path
import os

REPO_URL = "https://github.com/bqa100507-spec/T2ICount-Implementation.git"
REPO_DIR = Path("/content/T2ICount-Implementation")
BRANCH = "rich-prompt-alignment"

DRIVE_ASSET_ROOT = Path("/content/drive/MyDrive/T2ICount-assets")
ASSET_ROOT = Path("/content/T2ICount-assets")

PROMPT_BANK = DRIVE_ASSET_ROOT / "prompts/fsc147_train1000_seed3407_gemma4_v3.json"
OUTPUT_ROOT = DRIVE_ASSET_ROOT / "checkpoints/rich_alignment"

print("Repo:", REPO_DIR)
print("Drive assets:", DRIVE_ASSET_ROOT)
print("Local assets:", ASSET_ROOT)
print("Prompt bank:", PROMPT_BANK)
print("Output root:", OUTPUT_ROOT)


Repo: /content/T2ICount-Implementation
Drive assets: /content/drive/MyDrive/T2ICount-assets
Local assets: /content/T2ICount-assets
Prompt bank: /content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed3407_gemma4_v3.json
Output root: /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_alignment


In [4]:
%cd /content/T2ICount-Implementation
!git pull origin rich-prompt-alignment

[Errno 2] No such file or directory: '/content/T2ICount-Implementation'
/content
fatal: not a git repository (or any of the parent directories): .git


In [5]:
# 4. Clone / checkout the alignment branch
import subprocess

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)

print("\nHEAD:")
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)



HEAD:


CompletedProcess(args=['git', '-C', '/content/T2ICount-Implementation', 'log', '-1', '--oneline'], returncode=0)

## Dependency install

Chạy cell này **trước khi bật offline mode**, vì `requirements-colab.txt` có package cài trực tiếp từ GitHub.


In [6]:
# 5. Install Colab dependencies
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")],
    check=True,
)
print("Dependencies installed.")


Dependencies installed.


## Asset preparation

Drive của project hiện giữ một phần asset ở dạng nén:

- CLIP pretrained có thể nằm trong archive thay vì `pretrained/clip-vit-large-patch14/`.
- FSC147 image hiện có file `datasets/FSC147/FSC147_384_V2.zip` thay vì thư mục `images_384_VarV2/`.

Cell dưới tự xử lý cả hai trường hợp:

1. ưu tiên asset đã giải nén nếu có;
2. nếu không, tìm archive phù hợp;
3. với FSC147, **chỉ giải nén subtree `images_384_VarV2/`** thay vì giải nén toàn bộ file zip 1.5 GB;
4. copy metadata cần thiết sang `/content/T2ICount-assets`.

Không xóa hay sửa asset gốc trên Google Drive.


In [7]:
# 6. Prepare required alignment assets, including CLIP + FSC147 zip fallbacks
import shutil
import zipfile
from pathlib import Path

local_clip = ASSET_ROOT / "pretrained/clip-vit-large-patch14"
local_images = ASSET_ROOT / "datasets/FSC147/images_384_VarV2"
local_meta = ASSET_ROOT / "datasets/FSC147/FSC_147"

local_clip.parent.mkdir(parents=True, exist_ok=True)
local_images.parent.mkdir(parents=True, exist_ok=True)
local_meta.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# A. CLIP asset
# ---------------------------------------------------------------------
drive_clip = DRIVE_ASSET_ROOT / "pretrained/clip-vit-large-patch14"

def archive_contains_clip(zip_path):
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            return any(
                "clip-vit-large-patch14/config.json" in name.replace("\\", "/")
                for name in zf.namelist()
            )
    except (zipfile.BadZipFile, OSError):
        return False

def extract_named_subtree(zip_path, marker, destination):
    """
    Extract only files under a named subtree, regardless of any archive prefix.

    Example:
      marker='images_384_VarV2/'
      foo/bar/images_384_VarV2/123.jpg
        -> destination/123.jpg
    """
    destination.mkdir(parents=True, exist_ok=True)
    extracted = 0

    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            normalized = member.filename.replace("\\", "/")
            pos = normalized.find(marker)
            if pos < 0 or member.is_dir():
                continue

            relative = normalized[pos + len(marker):]
            if not relative:
                continue

            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)

            with zf.open(member, "r") as src_f, target.open("wb") as dst_f:
                shutil.copyfileobj(src_f, dst_f)
            extracted += 1

    if extracted == 0:
        raise RuntimeError(
            f"Archive {zip_path} does not contain subtree {marker!r}"
        )

    print(f"Extracted {extracted} files from {marker} in: {zip_path}")
    return extracted

if local_clip.exists():
    print("Local CLIP already exists:", local_clip)

elif drive_clip.exists():
    print("Copying unpacked CLIP asset from Drive...")
    shutil.copytree(drive_clip, local_clip)

else:
    print("Unpacked Drive CLIP not found. Searching zip archives...")

    likely = [
        Path("/content/drive/MyDrive/T2ICount-assets.zip"),
        Path("/content/drive/MyDrive/pretrained.zip"),
        DRIVE_ASSET_ROOT / "pretrained.zip",
    ]
    candidates = [p for p in likely if p.is_file()]

    mydrive = Path("/content/drive/MyDrive")
    for pattern in ("*.zip", "*/*.zip", "*/*/*.zip", "*/*/*/*.zip"):
        for p in mydrive.glob(pattern):
            if p not in candidates:
                candidates.append(p)

    print(f"Found {len(candidates)} zip candidate(s).")

    clip_archive = None
    for p in candidates:
        print("Checking:", p)
        if archive_contains_clip(p):
            clip_archive = p
            break

    if clip_archive is None:
        raise FileNotFoundError(
            "Could not find an unpacked CLIP directory or a zip archive containing "
            "'clip-vit-large-patch14/config.json'."
        )

    extract_named_subtree(
        clip_archive,
        marker="clip-vit-large-patch14/",
        destination=local_clip,
    )

# ---------------------------------------------------------------------
# B. FSC147 image asset
# Drive layout has FSC147_384_V2.zip rather than an unpacked images folder.
# ---------------------------------------------------------------------
drive_fsc_root = DRIVE_ASSET_ROOT / "datasets/FSC147"
drive_images = drive_fsc_root / "images_384_VarV2"
drive_image_zip = drive_fsc_root / "FSC147_384_V2.zip"
drive_meta = drive_fsc_root / "FSC_147"

if local_images.exists() and any(local_images.iterdir()):
    print("Local FSC147 image directory already exists:", local_images)

elif drive_images.exists():
    print("Copying unpacked FSC147 384 images from Drive...")
    shutil.copytree(drive_images, local_images)

elif drive_image_zip.is_file():
    print("Unpacked FSC147 image directory not found.")
    print("Extracting images_384_VarV2 only from:", drive_image_zip)

    extracted_count = extract_named_subtree(
        drive_image_zip,
        marker="images_384_VarV2/",
        destination=local_images,
    )
    print("FSC147 image files extracted:", extracted_count)

else:
    raise FileNotFoundError(
        "Could not find either:\n"
        f"  {drive_images}\n"
        "or\n"
        f"  {drive_image_zip}"
    )

# ---------------------------------------------------------------------
# C. FSC147 metadata
# ---------------------------------------------------------------------
required_meta = [
    drive_meta / "Train_Test_Val_FSC_147.json",
    drive_meta / "ImageClasses_FSC147.txt",
]

for p in required_meta:
    if not p.exists():
        raise FileNotFoundError(f"Missing required Drive dataset metadata: {p}")

for src_file in required_meta:
    dst_file = local_meta / src_file.name
    if not dst_file.exists():
        shutil.copy2(src_file, dst_file)

# Final checks
if not local_clip.is_dir():
    raise RuntimeError(f"CLIP preparation failed: {local_clip}")

if not local_images.is_dir():
    raise RuntimeError(f"FSC147 image preparation failed: {local_images}")

jpg_count = sum(1 for _ in local_images.glob("*.jpg"))
print("\nRequired alignment assets are ready.")
print("CLIP:", local_clip)
print("Images:", local_images, "jpg_count=", jpg_count)
print("Metadata:", local_meta)


Unpacked Drive CLIP not found. Searching zip archives...
Found 25 zip candidate(s).
Checking: /content/drive/MyDrive/T2ICount-assets.zip
Extracted 15 files from clip-vit-large-patch14/ in: /content/drive/MyDrive/T2ICount-assets.zip
Unpacked FSC147 image directory not found.
Extracting images_384_VarV2 only from: /content/drive/MyDrive/T2ICount-assets/datasets/FSC147/FSC147_384_V2.zip
Extracted 6146 files from images_384_VarV2/ in: /content/drive/MyDrive/T2ICount-assets/datasets/FSC147/FSC147_384_V2.zip
FSC147 image files extracted: 6146

Required alignment assets are ready.
CLIP: /content/T2ICount-assets/pretrained/clip-vit-large-patch14
Images: /content/T2ICount-assets/datasets/FSC147/images_384_VarV2 jpg_count= 6146
Metadata: /content/T2ICount-assets/datasets/FSC147/FSC_147


## CLIP processor config

T2ICount's local CLIP asset trước đây thiếu `preprocessor_config.json`.

Cell này chỉ tạo file khi nó chưa có, và tự loại UTF-8 BOM nếu file được tạo từ Windows PowerShell.


In [8]:
# 7. Ensure preprocessor_config.json exists and is UTF-8 without BOM
import json

processor_path = ASSET_ROOT / "pretrained/clip-vit-large-patch14/preprocessor_config.json"

processor_config = {
    "crop_size": 224,
    "do_center_crop": True,
    "do_normalize": True,
    "do_resize": True,
    "feature_extractor_type": "CLIPFeatureExtractor",
    "image_mean": [0.48145466, 0.4578275, 0.40821073],
    "image_std": [0.26862954, 0.26130258, 0.27577711],
    "resample": 3,
    "size": 224,
}

if not processor_path.exists():
    processor_path.write_text(
        json.dumps(processor_config, indent=2),
        encoding="utf-8",
    )
    print("Created:", processor_path)
else:
    raw = processor_path.read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        text = processor_path.read_text(encoding="utf-8-sig")
        processor_path.write_text(text, encoding="utf-8")
        print("Removed UTF-8 BOM:", processor_path)
    else:
        print("Processor config already exists:", processor_path)

print("First bytes:", processor_path.read_bytes()[:8])


Created: /content/T2ICount-assets/pretrained/clip-vit-large-patch14/preprocessor_config.json
First bytes: b'{\n  "cro'


In [9]:
# 8. Enable offline model loading AFTER repo/dependencies/assets are ready
os.environ["T2ICOUNT_ASSET_ROOT"] = str(ASSET_ROOT)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

assert PROMPT_BANK.is_file(), PROMPT_BANK
assert processor_path.is_file(), processor_path

print("Offline environment configured.")


Offline environment configured.


In [10]:
# 9. Direct CLIPProcessor sanity check
from transformers import CLIPProcessor

processor = CLIPProcessor.from_pretrained(
    str(ASSET_ROOT / "pretrained/clip-vit-large-patch14"),
    local_files_only=True,
)
print(processor)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPProcessor:
- image_processor: CLIPImageProcessor {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "CLIPImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}

- tokenizer: CLIPTokenizerFast(name_or_path='/content/T2ICount-assets/pretrained/clip-vit-large-patch14', vocab_size=49408, model_max_length=77, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	49406: AddedToken("<|startoftext|>", rstrip=False, lstrip=False, single_wor

# Validate only

Kỳ vọng:

- `Offline CLIP preflight passed.`
- CLIP image/text/projection load OK.
- subset fingerprint đúng.
- prompt-bank fingerprint đúng.
- split = 900 train / 100 validation.
- không ghi output training.


In [11]:
# 10. Validate-only
import subprocess, sys

validate_cmd = [
    sys.executable, "-u", "tools/train_rich_alignment.py",
    "--asset-root", str(ASSET_ROOT),
    "--prompt-bank", str(PROMPT_BANK),
    "--train-samples", "1000",
    "--subset-seed", "3407",
    "--split-seed", "3407",
    "--seed", "3407",
    "--device", "cuda",
    "--validate-only",
]

result = subprocess.run(validate_cmd, cwd=REPO_DIR, text=True)
print("RETURN CODE:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("validate-only failed")


RETURN CODE: 0


# Normalized smoke run — chạy trước full training

Chỉ train:

- 1 epoch FFN nhưng tối đa **1 batch**
- 1 epoch adapter nhưng tối đa **1 batch**

Khác biệt duy nhất so với raw smoke trước là:

```text
--normalize-embeddings
```

Khi flag này bật, final image/text embeddings được L2-normalize trước khi tính Euclidean distance.
Margin vẫn giữ nguyên `1.0`.

Mục tiêu chính là kiểm tra:

- positive / negative normalized Euclidean distance
- separation gap
- pairwise alignment accuracy
- margin violation rate
- retrieval R@1

**Chưa chạy full 10+10 sau cell này.**


In [12]:
# 11. Normalized smoke alignment run
import subprocess, sys

SMOKE_DIR = OUTPUT_ROOT / "smoke_norm_m12"

if SMOKE_DIR.exists():
    print("WARNING: normalized smoke output already exists:", SMOKE_DIR)
    print("If this directory belongs to a previous incompatible run, rename/delete it manually first.")

smoke_cmd = [
    sys.executable, "-u", "tools/train_rich_alignment.py",
    "--asset-root", str(ASSET_ROOT),
    "--prompt-bank", str(PROMPT_BANK),
    "--output-dir", str(SMOKE_DIR),
    "--train-samples", "1000",
    "--subset-seed", "3407",
    "--split-seed", "3407",
    "--seed", "3407",
    "--ffn-epochs", "1",
    "--adapter-epochs", "1",
    "--batch-size", "2",
    "--lr-ffn", "1e-4",
    "--lr-adapter", "1e-4",
    "--normalize-embeddings",
    "--margin", "1.2",
    "--dropout", "0.1",
    "--num-workers", "2",
    "--checkpoint-interval", "1",
    "--max-train-batches", "1",
    "--device", "cuda",
]

result = subprocess.run(smoke_cmd, cwd=REPO_DIR, text=True)
print("RETURN CODE:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Normalized smoke run failed")


If this directory belongs to a previous incompatible run, rename/delete it manually first.
RETURN CODE: 0


In [13]:
# 12. Inspect smoke summary
import json

summary_path = SMOKE_DIR / "alignment_summary.json"
if not summary_path.is_file():
    raise FileNotFoundError(summary_path)

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))


{
  "best_checkpoints": {
    "adapter": {
      "epoch": 1,
      "validation_overall_contrastive_loss": 1.001934548219045
    },
    "ffn": {
      "epoch": 1,
      "validation_overall_contrastive_loss": 1.0114599466323853
    }
  },
  "ffn_architecture": "minimal",
  "normalize_embeddings": true,
  "parameter_counts": {
    "frozen_clip": 427616513,
    "trainable_adapter": 1181184,
    "trainable_ffn": 1181184
  },
  "provenance": {
    "config_fingerprint": "sha256:f50bee40f9e92b8732786fec214549780340d5301fd66df31e27bfc02d2509ec",
    "ffn_architecture": "minimal",
    "normalize_embeddings": true,
    "prompt_bank_fingerprint": "sha256:c3ec587b37fd9f351cdedbce888b26e0ba7a020de956c4ef18202a1268e7e32f",
    "source_subset_fingerprint": "sha256:dd96b36bf15013e194b1a8ece06452a19822aae028fc00c0de019cbb7a311f24",
    "split_fingerprint": "sha256:3c4091f0f8558302f98ffbe808a25d28d9e7b428d59e50f1b6e62c787ad5e2a0",
    "training_distance": "l2_normalized_euclidean"
  },
  "stage0_clip": {

In [14]:
# 12b. Confirm normalized experiment provenance
prov = summary.get("provenance", {})
config_path = SMOKE_DIR / "alignment_config.json"

if config_path.is_file():
    cfg = json.loads(config_path.read_text(encoding="utf-8"))
    print("normalize_embeddings:", cfg.get("normalize_embeddings"))
    print("training_distance:", cfg.get("training_distance"))
else:
    print("alignment_config.json not found:", config_path)

print("Expected:")
print("normalize_embeddings = True")
print("training_distance = l2_normalized_euclidean")


normalize_embeddings: None
training_distance: None
Expected:
normalize_embeddings = True
training_distance = l2_normalized_euclidean


In [15]:
# 13. Compact stage view
for stage_name in ("stage0_clip", "stage1_ffn", "stage2_adapter"):
    stage = summary.get(stage_name)
    print("\n" + "=" * 80)
    print(stage_name)
    print("=" * 80)
    if stage is None:
        print("Missing stage in summary")
        continue

    for mode in ("overall", "class", "detailed", "generalized"):
        metrics = stage.get(mode)
        if metrics is None:
            continue
        print(f"\n[{mode}]")
        for key, value in metrics.items():
            if isinstance(value, (int, float, str, bool)):
                print(f"{key}: {value}")



stage0_clip

[overall]
contrastive_loss: 0.759502092997233
distance_distribution_aggregation: concatenated_class_detailed_generalized_distance_tensors
fraction_below_0_8: 0.0
fraction_below_0_9: 0.0
fraction_below_1_0: 0.0
fraction_below_1_1: 0.0
fraction_below_1_2: 0.0
fraction_below_1_25: 0.006666666828095913
fraction_below_1_3: 0.14666666090488434
fraction_below_1_35: 0.6466666460037231
fraction_below_1_4: 0.9700000286102295
fraction_below_1_45: 1.0
fraction_below_1_5: 1.0
margin_violation_rate: 0.0
mean_negative_euclidean_distance: 1.33543860912323
mean_positive_euclidean_distance: 1.2320321400960286
negative_cosine_similarity: 0.1077382539709409
negative_count_below_margin: 0
negative_fraction_below_margin: 0.0
negative_loss: 0.0
pairwise_alignment_accuracy: 0.9900000095367432
positive_cosine_similarity: 0.24049790700276694
positive_loss: 1.519004225730896
retrieval_aggregation: mean_across_prompt_modes
retrieval_class_aware_r_at_1: 0.8833333333333333
retrieval_r_at_1: 0.71
retri

## STOP HERE AFTER NORMALIZED SMOKE

Trước khi chạy full, kiểm tra Stage 0/1/2.

Với normalized embeddings:

\[
0 \le d \le 2
\]

nên margin `m=1` phải có ý nghĩa thực tế hơn raw Euclidean trước đó.

Điều cần xem:

- `margin_violation_rate` có **> 0** hay không;
- `separation_gap` có giữ dương hay collapse về 0;
- `pairwise_alignment_accuracy` có còn cao hơn random;
- retrieval có còn giữ được cấu trúc thay vì rơi về ~1–2%.

Nếu FFN/adapter vẫn collapse dù negative hinge đã hoạt động, **không chạy full 10+10**; khi đó cần kiểm tra kiến trúc/learning rate trước.


# Full 900/100 alignment — chỉ chạy sau khi smoke được duyệt

- 10 epochs FFN
- 10 epochs adapter
- batch size 32
- checkpoints mỗi epoch
- output lưu trên Drive


In [16]:
# 14. FULL RUN — intentionally disabled by default
import subprocess, sys

FULL_DIR = OUTPUT_ROOT / "run_norm_01"

full_cmd = [
    sys.executable, "-u", "tools/train_rich_alignment.py",
    "--asset-root", str(ASSET_ROOT),
    "--prompt-bank", str(PROMPT_BANK),
    "--output-dir", str(FULL_DIR),
    "--train-samples", "1000",
    "--subset-seed", "3407",
    "--split-seed", "3407",
    "--seed", "3407",
    "--ffn-epochs", "10",
    "--adapter-epochs", "10",
    "--batch-size", "32",
    "--lr-ffn", "1e-4",
    "--lr-adapter", "1e-4",
    "--normalize-embeddings",
    "--margin", "1.0",
    "--dropout", "0.1",
    "--num-workers", "2",
    "--checkpoint-interval", "1",
    "--device", "cuda",
]

print("FULL RUN COMMAND:")
print(" ".join(map(str, full_cmd)))
print("\nThis cell does NOT start training automatically.")
print("After smoke review, uncomment the subprocess.run block below.")

# result = subprocess.run(full_cmd, cwd=REPO_DIR, text=True)
# print("RETURN CODE:", result.returncode)
# if result.returncode != 0:
#     raise RuntimeError("Full alignment run failed")


FULL RUN COMMAND:
/usr/bin/python3 -u tools/train_rich_alignment.py --asset-root /content/T2ICount-assets --prompt-bank /content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed3407_gemma4_v3.json --output-dir /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_alignment/run_norm_01 --train-samples 1000 --subset-seed 3407 --split-seed 3407 --seed 3407 --ffn-epochs 10 --adapter-epochs 10 --batch-size 32 --lr-ffn 1e-4 --lr-adapter 1e-4 --normalize-embeddings --margin 1.0 --dropout 0.1 --num-workers 2 --checkpoint-interval 1 --device cuda

This cell does NOT start training automatically.
After smoke review, uncomment the subprocess.run block below.


In [ ]:
# 15. Read full-run summary after training
full_summary_path = FULL_DIR / "alignment_summary.json"

if full_summary_path.is_file():
    full_summary = json.loads(full_summary_path.read_text(encoding="utf-8"))
    print(json.dumps(full_summary, indent=2))
else:
    print("Full summary not found yet:", full_summary_path)


Full summary not found yet: /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_alignment/run_norm_01/alignment_summary.json


In [19]:
%%bash
set -euo pipefail
cd /content/T2ICount-Implementation
test "$(git branch --show-current)" = "rich-prompt-alignment"

export T2ICOUNT_ASSET_ROOT=/content/T2ICount-assets
export HF_HUB_OFFLINE=1
export TRANSFORMERS_OFFLINE=1

python -u tools/train_rich_alignment.py \
  --asset-root "$T2ICOUNT_ASSET_ROOT" \
  --prompt-bank /content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed3407_gemma4_v3.json \
  --output-dir /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_alignment/stageA_figure3bn_5e_m13 \
  --train-samples 1000 \
  --ffn-architecture figure3_bn \
  --stage-a-only \
  --ffn-epochs 5 \
  --adapter-epochs 1 \
  --batch-size 32 \
  --lr-ffn 1e-4 \
  --normalize-embeddings \
  --margin 1.3 \
  --dropout 0.1 \
  --seed 3407 \
  --subset-seed 3407 \
  --split-seed 3407 \
  --num-workers 2 \
  --device cuda

Offline CLIP preflight passed.
CLIPProcessor=OK
CLIPTokenizer=OK vocab_size=49408
CLIPModel=OK loading_info={"missing_keys": [], "unexpected_keys": [], "mismatched_keys": [], "error_msgs": []}
CLIPTextEncoder=OK hidden_size=768
CLIPVisionEncoder=OK hidden_size=1024
CLIPTextProjection=OK shape=(768, 768)
CLIPVisualProjection=OK shape=(768, 1024)
parameter_counts={"frozen_clip": 427616513, "trainable_ffn": 1182720, "trainable_adapter": 1181184}
source_subset_fingerprint=sha256:dd96b36bf15013e194b1a8ece06452a19822aae028fc00c0de019cbb7a311f24
prompt_bank_fingerprint=sha256:c3ec587b37fd9f351cdedbce888b26e0ba7a020de956c4ef18202a1268e7e32f
split_fingerprint=sha256:3c4091f0f8558302f98ffbe808a25d28d9e7b428d59e50f1b6e62c787ad5e2a0
alignment_train_count=900
alignment_validation_count=100
normalize_embeddings=true
training_distance=l2_normalized_euclidean
ffn_architecture=figure3_bn
{"best_epoch": 1, "best_validation_overall_contrastive_loss": 0.9010090827941895, "epoch": 1, "stage": "ffn", "train

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
